# Dual-Center Radial Compliance Analyzer (RCA-2) 12/25

This notebook measures **radial compositional compliance** in images by evaluating how visual mass organizes around two distinct coordinate frames:

1. **The global image frame**
2. **The detected mass center of the image**

It is designed to answer a *single, precise question*:

> **Where does radial structure actually exist in the image, and which coordinate system does it stabilize around?**

It does **not** attempt to answer questions of semantic importance, narrative focus, or what a human viewer “cares about.” Those are separate problems.

---

## What RCA-2 Measures

RCA-2 evaluates radial organization in **two coordinate frames**:

- **Frame-centered radial compliance (`RC_f`)**  
  Measures how closely the image’s mass distribution follows an ideal radial decay  
  **around the geometric center of the frame**.  
  This captures *global compositional priors* and default stabilization behavior.

- **Mass-centered radial compliance (`RC_s`)**  
  Measures how closely the image’s mass distribution follows an ideal radial decay  
  **around the centroid of the detected mass mask**.  
  This captures whether radial organization is anchored to the mass itself.

- **Delta radial compliance (`dRC = RC_s − RC_f`)**  
  Indicates whether radial structure aligns more strongly with the mass (`dRC > 0`)  
  or with the global frame (`dRC < 0`).

Together, these values distinguish **frame-dominant**, **mass-dominant**, and  
**dual-center** radial behavior.

---

## What This Notebook Explicitly Does *Not* Answer

Two common questions are often conflated:

1. **Where is the subject?**
2. **Where does radial structure exist?**

These are **not the same question**.

This notebook answers **#2 only**.

A subject may exist without supporting radial organization.  
A radial field may exist without being tied to a semantic subject.

RCA-2 makes no claim about *importance*, *meaning*, or *intent*—only about  
**structural stability**.

---

## Radial Eligibility (Critical Constraint)

**Mass-centered radial compliance (`RC_s`) is not automatically meaningful.**

A detected mass is only considered **radially eligible** if its shape plausibly  
supports radial organization. This requires:

- **Compactness** (mass is not elongated or fragmented)
- **Isotropy** (no dominant angular bias)
- **Low angular variance** (consistent distribution around the center)
- **Inside → outside continuity**

When these conditions fail, the notebook flags the image as **radially ineligible**, and:

> **`RC_s` should not be interpreted.**

This prevents false conclusions such as *“the subject is radial”* when the geometry  
does not support that claim.

---

## Radial Default Candidate (RDC) Gate

To avoid conflating **default model behavior** with **intentional or adversarial  
compositions**, the notebook introduces a **Radial Default Candidate (RDC)** concept.

RDC images are those that plausibly reflect a model’s *primary compositional prior*  
under neutral prompting—typically global, frame-aligned stabilization.

Images exhibiting:

- intentional asymmetry  
- competing attractors  
- field-engineered layouts  
- deliberate off-center strategies  

are **retained for inspection**, but **excluded from default measurements**.

Radial collapse, in this framework, is **not “about the subject.”**  
It is about **where the model finds stability**.

---

## What the Notebook Visualizes

To make the analysis legible, the notebook shows:

- Binary masks and overlays (frame center vs mass center)
- Radial ring visualizations
- Radial mass profiles with fitted exponential ideals
- Angular variance profiles
- Radial eligibility score and classification label

These visuals are **diagnostic**, not illustrative.

---

## Exported Metrics (Column Definitions)

The batch and single-image outputs use the following fields:

- **`frame_radial_compliance` (`RC_f`)**  
  Radial compliance around the image center.

- **`mass_radial_compliance` (`RC_s`)**  
  Radial compliance around the detected mass centroid  
  *(only valid if radially eligible)*.

- **`delta_radial_compliance` (`dRC`)**  
  Difference between mass-centered and frame-centered compliance  
  (`RC_s − RC_f`).

- **`frame_jsd` (`JSD_f`)**  
  Jensen–Shannon divergence between the frame-centered radial profile  
  and its ideal exponential decay.

- **`mass_jsd` (`JSD_s`)**  
  Jensen–Shannon divergence for the mass-centered radial profile.

- **`frame_alpha` (`alpha_f`)**  
  Fitted exponential decay parameter for the frame-centered profile.

- **`mass_alpha` (`alpha_s`)**  
  Fitted exponential decay parameter for the mass-centered profile.

- **`delta_x` (`dx`)**  
  Normalized horizontal offset between frame center and mass center.

- **`delta_y` (`dy`)**  
  Normalized vertical offset between frame center and mass center.

- **`delta_r` (`dr`)**  
  Euclidean distance between centers (`sqrt(dx² + dy²)`).

- **`radial_eligibility` (`E_s`)**  
  Scalar score indicating whether the mass shape plausibly supports  
  radial organization.

When `radial_eligibility` is low, mass-centered metrics are flagged and should be  
treated as **diagnostic only**, not interpretive.

---

## Bottom Line

RCA-2 is a **structural instrument**, not a semantic one.

It does not ask *what the subject is* or *why it matters*.  
It asks:

> **Where does radial order emerge, and which coordinate frame does it obey?**

That distinction is the entire point of the notebook.

1.

In [ ]:
!pip -q install opencv-python pillow numpy matplotlib pandas scipy

import os, io, zipfile
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.spatial.distance import jensenshannon
from google.colab import files

2.

In [ ]:
def pil_to_np_rgb(pil_img: Image.Image) -> np.ndarray:
    pil_img = pil_img.convert("RGB")
    return np.array(pil_img, dtype=np.uint8)

def read_image_rgb(path: str) -> np.ndarray:
    """
    Canonical decode for RCA-2:
    - PIL decode
    - EXIF transpose (fixes rotated iPhone / export cases)
    - RGB uint8 ndarray
    """
    im = Image.open(path)
    im = ImageOps.exif_transpose(im)
    im = im.convert("RGB")
    return np.array(im, dtype=np.uint8)

def show(img_rgb, title="", figsize=(6,6)):
    plt.figure(figsize=figsize)
    plt.imshow(img_rgb)
    plt.axis("off")
    if title: plt.title(title)
    plt.show()

3.

In [ ]:
def mask_from_otsu(img_rgb: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    _, th = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if th.mean() > 127:
        th = 255 - th
    return (th > 0).astype(np.uint8)

def mask_from_edges(img_rgb: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5,5), 0)
    edges = cv2.Canny(gray, 50, 150)
    edges = cv2.dilate(edges, np.ones((3,3), np.uint8), iterations=1)
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, np.ones((7,7), np.uint8))

    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros_like(gray, dtype=np.uint8)
    cv2.drawContours(mask, cnts, -1, 255, thickness=cv2.FILLED)
    return (mask > 0).astype(np.uint8)

def keep_largest_component(mask: np.ndarray) -> np.ndarray:
    mask = mask.astype(np.uint8)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num <= 1:
        return mask
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    return (labels == largest).astype(np.uint8)

def mask_hybrid_field(img_rgb: np.ndarray) -> np.ndarray:
    m1 = mask_from_edges(img_rgb)
    m2 = mask_from_otsu(img_rgb)
    m = np.clip(m1 + m2, 0, 1).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((9,9), np.uint8))
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5,5), np.uint8))
    return keep_largest_component(m)

4.

In [ ]:
def apply_border_penalty(mask: np.ndarray, strength: float = 2.0) -> np.ndarray:
    h, w = mask.shape
    yy, xx = np.mgrid[0:h, 0:w]
    d_edge = np.minimum.reduce([
        yy / max(h-1,1),
        (h-1-yy) / max(h-1,1),
        xx / max(w-1,1),
        (w-1-xx) / max(w-1,1),
    ])
    weight = np.clip(d_edge ** strength, 0, 1)
    return mask.astype(np.float64) * weight


def keep_compact_center_component(mask_bin: np.ndarray) -> np.ndarray:
    mask_bin = mask_bin.astype(np.uint8)
    num, labels, stats, cents = cv2.connectedComponentsWithStats(mask_bin, 8)
    if num <= 1:
        return mask_bin

    h, w = mask_bin.shape
    fx, fy = (w-1)/2, (h-1)/2

    best_i, best_score = None, -1
    for i in range(1, num):
        area = stats[i, cv2.CC_STAT_AREA]
        x,y,wc,hc,_ = stats[i]
        touches_edge = (x==0) or (y==0) or (x+wc>=w-1) or (y+hc>=h-1)
        dist = np.hypot(cents[i][0]-fx, cents[i][1]-fy)
        compactness = area / (wc*hc + 1e-9)

        # Reduced penalty for edge-touching if component is large and centered
        edge_penalty = 0.5 if touches_edge else 1.0
        score = (area * compactness * edge_penalty) / (1.0 + dist)

        if score > best_score:
            best_score = score
            best_i = i

    if best_i is None:
        return keep_largest_component(mask_bin)
    return (labels == best_i).astype(np.uint8)


def compute_mask(img_rgb: np.ndarray, mode: str = "field", border_strength: float = 1.5):
    """
    mode:
      - 'field'   : detects dominant field/plane mass (floors, water can win)
      - 'subject' : biases toward compact interior mass (reduces floor/water wins)

    border_strength: controls how aggressively to penalize edges
      - 1.0 = gentle (good for portraits)
      - 1.5 = moderate (recommended default)
      - 2.0+ = aggressive (may lose large subjects)
    """
    base = mask_hybrid_field(img_rgb)

    if mode == "field":
        return base.astype(np.uint8), base.astype(np.float64)

    if mode == "subject":
        weighted = apply_border_penalty(base, strength=border_strength)

        # Binarize with adaptive threshold
        bw = (weighted > 0.12).astype(np.uint8)  # Lowered from 0.15
        bw = keep_compact_center_component(bw)

        # Safety: if subject mode produces empty/tiny mask, fall back to field
        mask_area = bw.mean()
        if mask_area < 0.001:
            # Fall back to field mode
            return base.astype(np.uint8), base.astype(np.float64)

        weighted2 = apply_border_penalty(bw, strength=border_strength)
        return bw.astype(np.uint8), weighted2.astype(np.float64)

    raise ValueError("mode must be 'field' or 'subject'")

5.

In [ ]:
# =========================
# Radial Eligibility (E_s)
# =========================
# Goal: decide if RC_s is even interpretable for the chosen mask.
# No semantics. Pure geometry + angular dispersion.

def _mask_compactness(mask_bin: np.ndarray) -> float:
    # area / bbox area
    m = (mask_bin > 0).astype(np.uint8)
    ys, xs = np.where(m > 0)
    if len(xs) == 0:
        return 0.0
    area = float(len(xs))
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    bbox_area = float((x1 - x0 + 1) * (y1 - y0 + 1))
    return area / (bbox_area + 1e-9)

def _mask_isotropy(mask_bin: np.ndarray) -> float:
    """
    Isotropy via eigenvalues of 2nd moment around centroid.
    Returns in [0,1], where 1 = perfectly isotropic (round-ish), 0 = line-like.
    """
    m = (mask_bin > 0).astype(np.uint8)
    ys, xs = np.where(m > 0)
    if len(xs) < 10:
        return 0.0
    cx = xs.mean()
    cy = ys.mean()
    X = np.stack([xs - cx, ys - cy], axis=0)  # 2 x N
    C = (X @ X.T) / (X.shape[1] + 1e-9)       # 2 x 2 covariance-like
    evals = np.linalg.eigvalsh(C)
    l1, l2 = float(evals[0]), float(evals[1])  # l2 >= l1
    if l2 <= 1e-9:
        return 0.0
    return np.clip(l1 / (l2 + 1e-9), 0.0, 1.0)

def radial_eligibility_score(mask_bin: np.ndarray,
                             angv: np.ndarray,
                             RCs: float,
                             w_ang: float = 0.35,
                             w_iso: float = 0.35,
                             w_comp: float = 0.30) -> dict:
    """
    E_s ∈ [0,1]. High = subject mask is a reasonable candidate for radial organization.
    Inputs:
      - mask_bin: subject mask (binary)
      - angv: angular variance profile (same one you already plot)
      - RCs: subject-centered radial compliance (1 - JSD)
    """
    comp = _mask_compactness(mask_bin)
    iso  = _mask_isotropy(mask_bin)

    # angular variance: 0=perfectly radial, 1=directional chaos.
    # we want "low angular variance" -> high eligibility
    if angv is None or len(angv) == 0:
        ang_mean = 1.0
    else:
        # focus on inner radii where "subject" lives (avoid edge noise)
        k = max(5, int(0.60 * len(angv)))
        ang_mean = float(np.nanmean(angv[:k]))
        if np.isnan(ang_mean):
            ang_mean = 1.0

    ang_term = 1.0 - np.clip(ang_mean, 0.0, 1.0)

    # eligibility is "shape can be radial" *and* (optionally) RCs isn't garbage.
    # keep RCs *lightly* in the loop so eligibility doesn't become a duplicate of RCs.
    E = (w_ang * ang_term) + (w_iso * iso) + (w_comp * comp)
    E = float(np.clip(E, 0.0, 1.0))

    # lightweight flags
    radially_ineligible = (E < 0.38) or (iso < 0.22) or (comp < 0.08)
    return {
        "E_s": E,
        "ang_mean": ang_mean,
        "compactness": float(comp),
        "isotropy": float(iso),
        "radially_ineligible": bool(radially_ineligible),
    }

def classify_radial_behavior(RCf: float, RCs: float, dRC: float, Es: float,
                            void_ratio: float = None,
                            compactness: float = None) -> str:
    """
    Taxonomic classification of radial behavior.

    Categories:
    1. Ineligible - geometry doesn't support radial interpretation
    2. Field-dominant (vignette) - radial structure in lighting/gradient, sparse subject
    3. Field-dominant (collapse) - global basin dominates over subject
    4. Subject-dominant - subject mass drives radial organization
    5. Dual-center / near-tie - both centers contribute equally
    6. Weak/unclear - eligible but low coherence
    """

    # Gate: Geometric eligibility
    if Es < 0.38:
        return "Mass present but radially ineligible (do not interpret RC_s)"

    # Eligible cases - now taxonomize by structure

    # VIGNETTE DETECTION: High void + near-tie + strong RC = field gradient dominance
    # This catches: lighting gradients, atmospheric vignettes, sparse subjects in radial fields
    if void_ratio is not None and void_ratio > 0.85:
        # Very sparse composition (>85% void)
        if abs(dRC) < 0.03 and max(RCf, RCs) > 0.75:
            # Near-perfect tie + strong RC = radial field, not subject
            return "Field-dominant radial (vignette/gradient, sparse subject)"

    # FIELD COLLAPSE: Frame center dominates (negative dRC)
    if dRC < -0.06:
        return "Field-dominant radial collapse (global basin wins)"

    # SUBJECT-DOMINANT: Mass center dominates (positive dRC)
    if dRC > 0.06:
        # Could subdivide further if needed:
        # - "Subject-dominant radial (compact)" if compactness > 0.7
        # - "Subject-dominant radial (diffuse)" if compactness < 0.5
        return "Subject-dominant radial (rings follow mass)"

    # DUAL-CENTER / NEAR-TIE: Both contribute, but NOT vignette case
    # (vignette was already caught above)
    if max(RCf, RCs) > 0.62:
        return "Radial present (dual-center / near-tie)"

    # WEAK: Passes eligibility but low RC
    return "Weak/unclear radial (eligible mask, low coherence)"


6.

In [ ]:
def frame_center(mask_shape):
    h, w = mask_shape[:2]
    return ( (w-1)/2.0, (h-1)/2.0 )

def centroid_from_mass(M: np.ndarray):
    h, w = M.shape
    ys, xs = np.where(M > 0)
    if len(xs) == 0:
        return frame_center(M.shape)
    weights = M[ys, xs].astype(np.float64)
    S = weights.sum() + 1e-12
    cx = (xs * weights).sum() / S
    cy = (ys * weights).sum() / S
    return (cx, cy)

def max_corner_radius(cx, cy, h, w):
    corners = [(0,0), (0,h-1), (w-1,0), (w-1,h-1)]
    return max(np.hypot(x-cx, y-cy) for x,y in corners) + 1e-9

def radial_profile_from_mass(M: np.ndarray, cx: float, cy: float, nbins: int = 60):
    h, w = M.shape
    ys, xs = np.where(M > 0)
    if len(xs) == 0:
        r_centers = np.linspace(0,1,nbins)
        p = np.ones(nbins) / nbins
        return r_centers, p, np.array([]), np.array([])

    dx = xs - cx
    dy = ys - cy
    r = np.sqrt(dx*dx + dy*dy)
    r_norm = r / max_corner_radius(cx, cy, h, w)
    theta = np.arctan2(dy, dx)

    bins = np.linspace(0, 1, nbins+1)
    p = np.zeros(nbins, dtype=np.float64)
    weights = M[ys, xs].astype(np.float64)

    idx = np.clip(np.searchsorted(bins, r_norm, side="right") - 1, 0, nbins-1)
    for i, wgt in zip(idx, weights):
        p[i] += wgt

    p = p / (p.sum() + 1e-12)
    r_centers = (bins[:-1] + bins[1:]) / 2.0
    return r_centers, p, r_norm, theta

def angular_variance_by_radius(r_norm: np.ndarray, theta: np.ndarray, nbins: int = 30):
    if len(r_norm) == 0:
        return np.linspace(0,1,nbins), np.zeros(nbins)

    bins = np.linspace(0, 1, nbins+1)
    r_centers = (bins[:-1] + bins[1:]) / 2.0
    out = []

    for i in range(nbins):
        m = (r_norm >= bins[i]) & (r_norm < bins[i+1])
        t = theta[m]
        if len(t) < 10:
            out.append(np.nan); continue
        C = np.mean(np.cos(t))
        S = np.mean(np.sin(t))
        R = np.sqrt(C*C + S*S)
        out.append(1.0 - R)

    return r_centers, np.array(out, dtype=np.float64)

7.

In [ ]:
from scipy.optimize import minimize_scalar

def fit_exponential_ideal(r_centers: np.ndarray, p: np.ndarray):
    """
    Fit exponential decay profile using bounded optimization.

    Returns: (best_profile, best_alpha)
    """
    r = r_centers.astype(np.float64)
    p = p.astype(np.float64)

    # Define loss function
    def loss(alpha):
        q = np.exp(-alpha * r)
        q = q / (q.sum() + 1e-12)
        return np.mean((q - p)**2)

    # Optimize with wide bounds
    result = minimize_scalar(
        loss,
        bounds=(0.1, 200.0),  # Wide range to handle extreme cases
        method='bounded',
        options={'xatol': 0.01}  # Tolerance for alpha
    )

    # Get best alpha and construct profile
    best_alpha = float(result.x)
    q = np.exp(-best_alpha * r)
    q = q / (q.sum() + 1e-12)

    return q, best_alpha

def radial_compliance(p: np.ndarray, q: np.ndarray):
    jsd = float(jensenshannon(p + 1e-12, q + 1e-12))
    rc = max(0.0, min(1.0, 1.0 - jsd))
    return rc, jsd

8.

In [ ]:
def overlay_mask(img_rgb: np.ndarray, mask_bin: np.ndarray, alpha=0.45):
    out = img_rgb.astype(np.float32).copy()
    m = mask_bin.astype(bool)
    tint = np.zeros_like(out)
    tint[m] = np.array([255,255,255], dtype=np.float32)
    out = out*(1-alpha) + tint*alpha
    return out.astype(np.uint8)

def draw_point(img_rgb: np.ndarray, x: float, y: float, color=(255,0,0), r=8):
    out = img_rgb.copy()
    cv2.circle(out, (int(round(x)), int(round(y))), r, color, -1)
    cv2.circle(out, (int(round(x)), int(round(y))), r*2, color, 2)
    return out

def draw_rings(img_rgb: np.ndarray, cx: float, cy: float, n=22, color=(220,0,0), thickness=2):
    out = img_rgb.copy()
    h, w = out.shape[:2]
    R = max_corner_radius(cx, cy, h, w)
    for i in range(1, n+1):
        rad = int(round((i/(n+1)) * R))
        cv2.circle(out, (int(round(cx)), int(round(cy))), rad, color, thickness)
    return out

def deviation_heatmap(M: np.ndarray, cx: float, cy: float, r_centers: np.ndarray, q: np.ndarray):
    h, w = M.shape
    yy, xx = np.mgrid[0:h, 0:w]
    r = np.sqrt((xx-cx)**2 + (yy-cy)**2)
    r_norm = r / max_corner_radius(cx, cy, h, w)

    nb = len(r_centers)
    idx = np.clip((r_norm * nb).astype(int), 0, nb-1)
    expected = q[idx]
    dev = M - expected
    dev = gaussian_filter(dev, sigma=1.0)
    return dev

def plot_profile(r_centers, p, q, alpha, rc, jsd, title=""):
    plt.figure(figsize=(7,4))
    plt.plot(r_centers, p, label="Measured p(r)")
    plt.plot(r_centers, q, label=f"Ideal exp(-αr), α={alpha:.2f}")
    plt.xlabel("Normalized radius")
    plt.ylabel("Mass (normalized)")
    plt.title(f"{title} | RC={rc:.3f} (1-JSD), JSD={jsd:.3f}")
    plt.legend()
    plt.show()

def plot_heat(img_rgb, dev, title="Deviation heatmap"):
    plt.figure(figsize=(7,6))
    plt.imshow(img_rgb)
    plt.imshow(dev, alpha=0.55)
    plt.axis("off")
    plt.title(title)
    plt.show()

9.

In [ ]:
# =========================
# Centroid helper
# =========================

def centroid_from_mask(mask_bin: np.ndarray):
    """
    Returns (cx, cy) = mass centroid of a binary mask.
    This is a geometric centroid, NOT a semantic subject center.
    """
    m = (mask_bin > 0).astype(np.float32)
    total = m.sum()
    if total <= 1e-9:
        # fallback to frame center if mask is empty
        h, w = mask_bin.shape
        return (w / 2.0, h / 2.0)

    ys, xs = np.indices(mask_bin.shape)
    cx = float((xs * m).sum() / total)
    cy = float((ys * m).sum() / total)
    return cx, cy

10.

In [ ]:
# ============================================================
# Compatibility aliases (do NOT change these names)
# ============================================================

def radial_mass_profile(M, center, nbins=60):
    """
    Wrapper for radial_profile_from_mass.
    Unpacks center tuple into cx, cy.

    Returns: (r_centers, p)
    """
    cx, cy = center  # Unpack the tuple
    r_centers, p, r_norm, theta = radial_profile_from_mass(M, cx, cy, nbins=nbins)
    return r_centers, p


def angular_variance_profile(M, center, nbins=60):
    """
    Wrapper that combines radial_profile_from_mass + angular_variance_by_radius.

    Returns: ang_var (just the variance array, NOT r_centers)

    Note: r_centers is already available from radial_mass_profile(),
    so we only return the angular variance values.
    """
    cx, cy = center  # Unpack the tuple

    # Get r_norm and theta from radial profile
    _, _, r_norm, theta = radial_profile_from_mass(M, cx, cy, nbins=nbins)

    # Compute angular variance
    _, ang_var = angular_variance_by_radius(r_norm, theta, nbins=nbins)

    return ang_var  # Return ONLY the variance array


def fit_exp_alpha(p, r):
    """
    Wrapper for fit_exponential_ideal.

    Note: Swaps argument order - callers pass (p, r) but
    fit_exponential_ideal expects (r_centers, p).

    Returns: alpha (scalar, not tuple)
    """
    _, alpha = fit_exponential_ideal(r, p)  # Swap order, extract alpha
    return alpha


def ideal_exp_profile(r, alpha):
    """Generate ideal exponential profile"""
    r = np.asarray(r, dtype=float)
    return np.exp(-alpha * r)

11.

In [ ]:
# =========================
# RCA-2 Canonical Defaults
# =========================
DEFAULT_MASK_MODE = "subject"
DEFAULT_BORDER_STRENGTH = 1.5 #was 2.2
DEFAULT_NBINS = 60

def analyze_single(
    img_rgb: np.ndarray,
    mask_mode: str = DEFAULT_MASK_MODE,
    border_strength: float = DEFAULT_BORDER_STRENGTH,
    nbins: int = DEFAULT_NBINS
):
    # --- mask ---
    mask_bin, M = compute_mask(
        img_rgb, mode=mask_mode, border_strength=border_strength
    )
    h, w = mask_bin.shape

    # --- centers ---
    fx, fy = frame_center(mask_bin.shape)
    sx, sy = centroid_from_mask(mask_bin)

    # --- radial profiles ---
    r, pf = radial_mass_profile(M, (fx, fy), nbins=nbins)  # ← CORRECT ORDER
    _, ps = radial_mass_profile(M, (sx, sy), nbins=nbins)

    # --- ideal fits ---
    af = fit_exp_alpha(pf, r)
    aS = fit_exp_alpha(ps, r)
    qf = ideal_exp_profile(r, af)
    qs = ideal_exp_profile(r, aS)

    # --- compliance ---
    RCf, JSDf = radial_compliance(pf, qf)
    RCs, JSDs = radial_compliance(ps, qs)
    dRC = RCs - RCf

    # --- spatial deltas ---
    dx = (sx - fx) / max(w, 1)
    dy = (sy - fy) / max(h, 1)
    dr = float(np.hypot(dx, dy))

    # --- angular variance (subject-centered) ---
    angv = angular_variance_profile(M, (sx, sy), nbins=nbins)

    # --- eligibility ---
    elig = radial_eligibility_score(
        mask_bin=mask_bin,
        angv=angv,
        RCs=RCs
    )
    Es = elig["E_s"]

    # Calculate void_ratio for classification
    void_ratio = float(1.0 - mask_bin.mean())

    # Pass void_ratio to classifier
    label = classify_radial_behavior(
        RCf=RCf,
        RCs=RCs,
        dRC=dRC,
        Es=Es,
        void_ratio=void_ratio,
        compactness=float(elig.get("compactness", 0.0))
    )

    # =========================
    # Visualizations
    # =========================
    show(img_rgb, "Original")

    plt.figure(figsize=(6,6))
    plt.imshow(mask_bin, cmap="gray")
    plt.axis("off")
    plt.title(f"Mask ({mask_mode})")
    plt.show()

    overlay = overlay_mask(img_rgb, mask_bin)
    overlay = draw_point(overlay, fx, fy, color=(0,255,255))
    overlay = draw_point(overlay, sx, sy, color=(255,0,0))
    show(overlay, "Overlay: frame (cyan) + mass (red)", figsize=(8,6))

    show(draw_rings(img_rgb, fx, fy), "Rings @ frame center", figsize=(8,6))
    show(draw_rings(img_rgb, sx, sy), "Rings @ mass center", figsize=(8,6))

    plot_profile(r, pf, qf, af, RCf, JSDf, "Frame-centered profile")
    plot_profile(r, ps, qs, aS, RCs, JSDs, "Mass-centered profile")

    plt.figure(figsize=(7,4))
    plt.plot(r, angv)
    plt.ylim(0,1)
    plt.xlabel("Normalized radius")
    plt.ylabel("Angular variance")
    plt.title("Angular variance (mass-centered)")
    plt.show()

    summary = {
        "mask_mode": mask_mode,
        "RC_f": RCf,
        "JSD_f": JSDf,
        "alpha_f": af,
        "RC_s": RCs,
        "JSD_s": JSDs,
        "alpha_s": aS,
        "dRC": dRC,
        "dx": dx,
        "dy": dy,
        "dr": dr,
        "void_ratio": float(1.0 - mask_bin.mean()),
        "E_s": Es,
        "compactness": elig["compactness"],
        "isotropy": elig["isotropy"],
        "ang_mean": elig["ang_mean"],
        "radially_ineligible": elig["radially_ineligible"],
        "radial_label": label
    }

    print("Summary:")
    print(f"  Frame-centered radial compliance (RC_f): {summary['RC_f']}")
    print(f"  Mass-centered radial compliance (RC_s): {summary['RC_s']}")
    print(f"  ΔRC (mass − frame): {summary['dRC']}")
    print(f"  Radial eligibility (E_s): {summary['E_s']}")
    print(f"  Radial label: {summary['radial_label']}")

    # Optional: keep raw dump for debugging
    # for k, v in summary.items():
    #     print(f"  {k}: {v}")

    return summary

12.

In [ ]:
uploaded = files.upload()
name = next(iter(uploaded.keys()))
img_rgb = pil_to_np_rgb(Image.open(io.BytesIO(uploaded[name])))

# Choose mask mode:
#  - "field"   : floors/water can dominate
#  - "subject" : biases toward compact interior subject mass
summary = analyze_single(img_rgb)

df = pd.DataFrame([summary])

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

display(df)

In [ ]:
# ============================================
# Single-image result rendered as batch-style row
# ============================================

def single_summary_to_batch_row(summary: dict, filename: str = "SINGLE_IMAGE"):
    """
    Convert analyze_single() output dict into a
    one-row DataFrame matching batch CSV layout.
    """
    row = {
        "filename": filename,

        # --- Core radial metrics ---
        "RC_f": summary.get("RC_f"),
        "JSD_f": summary.get("JSD_f"),
        "alpha_f": summary.get("alpha_f"),

        "RC_s": summary.get("RC_s"),
        "JSD_s": summary.get("JSD_s"),
        "alpha_s": summary.get("alpha_s"),

        "dRC": summary.get("dRC"),

        # --- Spatial offsets ---
        "dx": summary.get("dx"),
        "dy": summary.get("dy"),
        "dr": summary.get("dr"),

        # --- Geometry / eligibility ---
        "void_ratio": summary.get("void_ratio"),
        "E_s": summary.get("E_s"),
        "compactness": summary.get("compactness"),
        "isotropy": summary.get("isotropy"),
        "ang_mean": summary.get("ang_mean"),
        "radially_ineligible": summary.get("radially_ineligible"),

        # --- Classification ---
        "radial_label": summary.get("radial_label"),
    }

    return pd.DataFrame([row])


# ---- render ----
df_single_row = single_summary_to_batch_row(
    summary,
    filename=os.path.basename(single_path) if "single_path" in globals() else "SINGLE_IMAGE"
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

display(df_single_row)

13.

In [ ]:
def compare_mask_modes(img_rgb):
    plt.figure(figsize=(14,4))
    plt.subplot(1,3,1); plt.imshow(img_rgb); plt.axis("off"); plt.title("Original")

    m_field, _ = compute_mask(img_rgb, mode="field")
    plt.subplot(1,3,2); plt.imshow(m_field*255, cmap="gray"); plt.axis("off"); plt.title("Mask: field")

    m_subj, _ = compute_mask(img_rgb, mode="subject", border_strength=2.2)
    plt.subplot(1,3,3); plt.imshow(m_subj*255, cmap="gray"); plt.axis("off"); plt.title("Mask: subject (border-biased)")
    plt.show()

compare_mask_modes(img_rgb)

13.5

In [ ]:
# --- (13.5) RDC gate (single-image only) ---
# Insert between your mask-compare cell (13) and batch intake (14)

if "summary" not in globals():
    raise RuntimeError("summary not found. Run the single-image analyze cell first.")

# Tunable thresholds (start conservative)
TAU_MASK_AREA = 0.001     # must have enough detected mass mask_area ≥ 0.001
EPS_ELIG      = 0.55      # eligibility must be decent (your example was 0.55), The detected mass must be compact enough, isotropic enough, low enough angular variance to plausibly support a radial interpretation
DELTA_R_MAX   = 0.12      # avoid strongly off-center engineered comps
RCF_MIN       = 0.70      # require meaningful frame radiality, strong frame-centered radial order
KAPPA_DRC_MAX = 0.03      # near-tie or frame-dominant (exclude strong subject-dominant), , frame-dominant

mask_area = 1.0 - float(summary.get("void_ratio", 1.0))
Es = float(summary.get("E_s", float("nan")))
dr = float(summary.get("dr", float("nan")))
RCf = float(summary.get("RC_f", float("nan")))
dRC = float(summary.get("dRC", float("nan")))

reasons = []
ok = True

if mask_area < TAU_MASK_AREA:
    ok = False; reasons.append(f"mask_area<{TAU_MASK_AREA:g}")
if not (Es == Es) or Es < EPS_ELIG:  # Es==Es checks NaN
    ok = False; reasons.append(f"E_s<{EPS_ELIG:g}")
if not (dr == dr) or dr > DELTA_R_MAX:
    ok = False; reasons.append(f"delta_r>{DELTA_R_MAX:g}")
if not (RCf == RCf) or RCf < RCF_MIN:
    ok = False; reasons.append(f"RC_f<{RCF_MIN:g}")
if not (dRC == dRC) or dRC > KAPPA_DRC_MAX:
    ok = False; reasons.append(f"dRC>{KAPPA_DRC_MAX:g} (subject-dominant / engineered)")

print("RDC Gate (single-image):")
print("  RDC =", bool(ok))
print("  mask_area =", mask_area)
print("  E_s =", Es)
print("  delta_r =", dr)
print("  RC_f =", RCf)
print("  dRC =", dRC)

if ok:
    print("  Reasons: passes default-prior plausibility filter.")
else:
    print("  Reasons:", "; ".join(reasons))
    print("  Note: non-RDC does NOT mean 'bad' — it means 'not default-prior measurement'.")

14.

In [ ]:
# --- (14) CLEAN batch intake (hard reset, unzip, skip Mac artifacts, recursive scan) ---

from scipy.optimize import minimize_scalar

import os, zipfile, shutil, hashlib
from pathlib import Path
from google.colab import files

BATCH_DIR = "/content/rca_batch"
IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}

def _is_bad_path(p: Path) -> bool:
    name = p.name
    if name.startswith("._") or name.startswith(".") or name.lower() == "thumbs.db":
        return True
    # Skip any path that lives under __MACOSX (common in zips made on macOS)
    if "__MACOSX" in p.parts:
        return True
    return False

def _md5(p: Path) -> str:
    h = hashlib.md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# HARD RESET so old zips/images can't accumulate across runs
if os.path.isdir(BATCH_DIR):
    shutil.rmtree(BATCH_DIR)
os.makedirs(BATCH_DIR, exist_ok=True)

print("Upload EITHER: (a) one .zip of images OR (b) multiple images directly.")
uploaded = files.upload()

# write uploads into BATCH_DIR
for fn, data in uploaded.items():
    out_path = os.path.join(BATCH_DIR, fn)
    with open(out_path, "wb") as f:
        f.write(data)

# unzip any zip(s) directly into BATCH_DIR, then remove the zip(s)
for fn in list(os.listdir(BATCH_DIR)):
    if fn.lower().endswith(".zip"):
        zpath = os.path.join(BATCH_DIR, fn)
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall(BATCH_DIR)
        os.remove(zpath)
        print("Unzipped + removed:", fn)

# build candidate list recursively
all_files = [p for p in Path(BATCH_DIR).rglob("*") if p.is_file()]
img_files = [p for p in all_files if (p.suffix.lower() in IMG_EXT) and (not _is_bad_path(p))]

print("Images found recursively:", len(img_files))
if len(img_files) == 0:
    paths = []
    print("Found usable images: 0")
else:
    # dedupe by file hash (prevents duplicates when zips include copies)
    seen = set()
    paths = []
    for p in sorted(img_files):
        h = _md5(p)
        if h in seen:
            continue
        seen.add(h)
        paths.append(str(p))

    print("Found usable images:", len(paths))
    print("Sample:", paths[:5])

15.

In [ ]:
# --- (15) Batch run (robust + consistent + mask-empty guard) ---
import os, traceback
import numpy as np
import pandas as pd
from tqdm import tqdm

MASK_MODE = DEFAULT_MASK_MODE
BORDER_STRENGTH = DEFAULT_BORDER_STRENGTH
NBINS = DEFAULT_NBINS
MASK_EMPTY_THRESH = 0.001  # raise to 0.005 if you want fewer "near-empty but noisy" cases

# --- FIXED: Validate paths without resetting ---
if "paths" not in globals():
    raise RuntimeError("❌ 'paths' not defined. Run Cell 29 to upload images first.")

if not isinstance(paths, list):
    raise RuntimeError("❌ 'paths' is not a list. Run Cell 29 again.")

if len(paths) == 0:
    print("⚠️  WARNING: paths list is empty. No images to process.")
    print("   Re-run Cell 29 to upload images.")

    # Create empty dataframe so downstream cells don't crash
    df = pd.DataFrame(columns=[
        "filename", "path", "mask_mode", "error", "trace",
        "RC_f", "JSD_f", "alpha_f", "RC_s", "JSD_s", "alpha_s",
        "dRC", "dx", "dy", "dr", "void_ratio",
        "E_s", "compactness", "isotropy", "ang_mean",
        "radially_ineligible", "radial_label", "mask_area"
    ])
    display(df)

else:
    print(f"✓ Found {len(paths)} images to analyze")

    def _safe_float(x):
        try:
            if x is None:
                return np.nan
            return float(x)
        except Exception:
            return np.nan

    def analyze_path_quick(path: str,
                           mask_mode: str = "subject",
                           border_strength: float = 1.5,
                           nbins: int = 60):

        out = {
            "filename": os.path.basename(path),
            "path": path,
            "mask_mode": mask_mode,
            "error": None,
            "trace": None,
        }

        try:
            img_rgb = read_image_rgb(path)

            mask_bin, M = compute_mask(img_rgb, mode=mask_mode, border_strength=border_strength)
            if mask_bin is None or M is None:
                raise ValueError("compute_mask returned None")

            mask_bin = np.asarray(mask_bin).astype(bool)
            M = np.asarray(M).astype(float)

            if mask_bin.ndim != 2 or M.ndim != 2:
                raise ValueError(f"mask_bin/M not 2D: {mask_bin.shape}, {M.shape}")
            if mask_bin.shape != M.shape:
                raise ValueError(f"mask_bin and M shape mismatch: {mask_bin.shape} vs {M.shape}")

            # --- key: mask-empty guard (prevents blank/meaningless plots later) ---
            mask_area = float(mask_bin.mean())
            void_ratio = float(1.0 - mask_area)
            out["mask_area"] = mask_area

            if mask_area < MASK_EMPTY_THRESH:
                # Keep the row valid, but mark all radial metrics unusable
                out.update({
                    "RC_f": np.nan, "JSD_f": np.nan, "alpha_f": np.nan,
                    "RC_s": np.nan, "JSD_s": np.nan, "alpha_s": np.nan,
                    "dRC": np.nan,
                    "dx": np.nan, "dy": np.nan, "dr": np.nan,
                    "void_ratio": void_ratio,
                    "E_s": np.nan, "compactness": np.nan, "isotropy": np.nan, "ang_mean": np.nan,
                    "radially_ineligible": True,
                    "radial_label": "MASK EMPTY (skip radial)"
                })
                return out

            # --- centers ---
            h, w = mask_bin.shape
            fx, fy = frame_center(mask_bin.shape)

            # IMPORTANT: use one "mass center" definition everywhere.
            # This should match your analyzer-style binary mask centroid.
            sx, sy = centroid_from_mask(mask_bin)

            # --- radial profiles ---
            rav, pf = radial_mass_profile(M, (fx, fy), nbins=nbins)  # ← SWAPPED
            _, ps   = radial_mass_profile(M, (sx, sy), nbins=nbins)

            pf = np.asarray(pf, dtype=float).ravel()
            ps = np.asarray(ps, dtype=float).ravel()
            rav = np.asarray(rav, dtype=float).ravel()

            af = _safe_float(fit_exp_alpha(pf, rav))
            aS = _safe_float(fit_exp_alpha(ps, rav))

            qf = np.asarray(ideal_exp_profile(rav, af), dtype=float).ravel()
            qS = np.asarray(ideal_exp_profile(rav, aS), dtype=float).ravel()

            # ← FIXED: radial_compliance returns (rc, jsd) tuple
            RCf_raw, JSDf_raw = radial_compliance(pf, qf)
            RCf = _safe_float(RCf_raw)
            JSDf = _safe_float(JSDf_raw)

            RCs_raw, JSDs_raw = radial_compliance(ps, qS)
            RCs = _safe_float(RCs_raw)
            JSDs = _safe_float(JSDs_raw)

            dRC = _safe_float(RCs - RCf)

            dx = (sx - fx) / max(w, 1)
            dy = (sy - fy) / max(h, 1)
            dr = float(np.hypot(dx, dy))

            # --- eligibility + label ---
            angv = angular_variance_profile(M, (sx, sy), nbins=nbins)
            elig = radial_eligibility_score(mask_bin=mask_bin, angv=angv, RCs=RCs)

            Es = _safe_float(elig.get("E_s", np.nan))
            label = classify_radial_behavior(
                RCf=RCf,
                RCs=RCs,
                dRC=dRC,
                Es=Es,
                void_ratio=void_ratio,  # ← ADD THIS
                compactness=_safe_float(elig.get("compactness", np.nan))  # ← ADD THIS (optional)
            )

            out.update({
                "RC_f": _safe_float(RCf),
                "JSD_f": _safe_float(JSDf),
                "alpha_f": _safe_float(af),

                "RC_s": _safe_float(RCs),
                "JSD_s": _safe_float(JSDs),
                "alpha_s": _safe_float(aS),

                "dRC": _safe_float(dRC),
                "dx": _safe_float(dx),
                "dy": _safe_float(dy),
                "dr": _safe_float(dr),
                "void_ratio": _safe_float(void_ratio),

                "E_s": _safe_float(Es),
                "compactness": _safe_float(elig.get("compactness", np.nan)),
                "isotropy": _safe_float(elig.get("isotropy", np.nan)),
                "ang_mean": _safe_float(elig.get("ang_mean", np.nan)),
                "radially_ineligible": bool(elig.get("radially_ineligible", False)),
                "radial_label": label,
            })
            return out

        except Exception as e:
            out["error"] = f"{type(e).__name__}: {e}"
            out["trace"] = traceback.format_exc(limit=8)
            return out

    # --- Execute batch processing ---
    try:
        rows = [analyze_path_quick(p, MASK_MODE, BORDER_STRENGTH, NBINS) for p in tqdm(paths)]
        df = pd.DataFrame(rows)

        print(f"\n✓ Batch processing complete")
        print(f"  Total rows: {len(df)}")

        if "error" in df.columns:
            error_count = int(df["error"].notna().sum())
            print(f"  Errors: {error_count}")
            if error_count > 0:
                print(f"\n⚠️  {error_count} images failed to process")
                print("\nTop errors:")
                display(df[df["error"].notna()][["filename", "error"]].head(5))

        # Display summary
        cols = [c for c in ["filename", "error", "RC_f", "RC_s", "dRC", "E_s", "radial_label", "mask_area"]
                if c in df.columns]

        print(f"\nFirst 20 rows:")
        display(df[cols].head(20) if cols else df.head(20))

    except Exception as e:
        print(f"❌ BATCH PROCESSING FAILED")
        print(f"   {type(e).__name__}: {e}")
        print("\nFull traceback:")
        traceback.print_exc()

        # Create empty df so downstream cells can run
        df = pd.DataFrame(columns=[
            "filename", "path", "mask_mode", "error", "trace",
            "RC_f", "JSD_f", "alpha_f", "RC_s", "JSD_s", "alpha_s",
            "dRC", "dx", "dy", "dr", "void_ratio",
            "E_s", "compactness", "isotropy", "ang_mean",
            "radially_ineligible", "radial_label", "mask_area"
        ])


16.

In [ ]:
# --- (16) Batch summary charts (safe even if many rows errored) ---

import matplotlib.pyplot as plt

if "df" not in globals():
    raise RuntimeError("df not defined. Run Cell 15 first.")

def hist(series, title, bins=25):
    s = series.dropna()
    if len(s) == 0:
        print(f"(skip) {title}: no data")
        return
    plt.figure(figsize=(7,4))
    plt.hist(s.values, bins=bins)
    plt.title(title)
    plt.show()

df_ok = df[df["error"].isna()].copy() if "error" in df.columns else df.copy()
print("Valid:", len(df_ok), "/", len(df))

if len(df_ok) == 0:
    print("No valid rows — top errors:")
    display(df["error"].value_counts().head(20))
else:
    for col, title in [
        ("RC_f", "Frame-centered radial compliance distribution"),
        ("RC_s", "Mass-centered radial compliance distribution"),
        ("dRC",  "ΔRC distribution (mass − frame)"),
        ("dr",   "Δr distribution (subject vs frame center)"),
        ("E_s",  "E_s distribution (radial eligibility)"),
    ]:
        if col in df_ok.columns:
            hist(df_ok[col], title, bins=25)

    if "radial_label" in df_ok.columns:
        print("\nRadial label counts:")
        display(df_ok["radial_label"].value_counts())

17.

In [ ]:
# --- Export results (canonical names) ---
from google.colab import files

if "df" not in globals():
    raise RuntimeError("df not defined. Run the batch cell first.")

df_export = df.rename(columns={
    "RC_f": "frame_radial_compliance",
    "RC_s": "mass_radial_compliance",
    "dRC":  "delta_radial_compliance",
    "JSD_f": "frame_jsd",
    "JSD_s": "mass_jsd",
    "alpha_f": "frame_alpha",
    "alpha_s": "mass_alpha",
    "dx": "delta_x",
    "dy": "delta_y",
    "dr": "delta_r",
    "E_s": "radial_eligibility",
})

OUT = "/content/rca2_batch_results.csv"
df_export.to_csv(OUT, index=False)
print("Saved:", OUT)
files.download(OUT)

18.

In [ ]:
def quicklook(paths, n=10, mask_mode="subject", border_strength=2.2):
    n = min(n, len(paths))
    if n == 0:
        print("No images."); return

    cols = 3
    plt.figure(figsize=(14, 4*n))
    for i in range(n):
        img = read_image_rgb(paths[i])
        mask_bin, M = compute_mask(img, mode=mask_mode, border_strength=border_strength)
        fx, fy = frame_center(mask_bin.shape)
        sx, sy = centroid_from_mask(mask_bin)
        overlay = overlay_mask(img, mask_bin)
        overlay = draw_point(overlay, fx, fy, color=(0,255,255))
        overlay = draw_point(overlay, sx, sy, color=(255,0,0))

        plt.subplot(n, cols, i*cols+1); plt.imshow(img); plt.axis("off"); plt.title(os.path.basename(paths[i]))
        plt.subplot(n, cols, i*cols+2); plt.imshow(mask_bin*255, cmap="gray"); plt.axis("off"); plt.title("Mask")
        plt.subplot(n, cols, i*cols+3); plt.imshow(overlay); plt.axis("off"); plt.title("Centers: frame(cyan), subject(red)")

    plt.tight_layout()
    plt.show()

quicklook(paths, n=8, mask_mode="subject", border_strength=2.2)

19.

In [ ]:
# --- (19) Render outputs for each image (self-contained; matches batch metrics) ---

import os, traceback
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

OUT_DIR = "/content/rca2_outputs"

def ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

ensure_dir(OUT_DIR)

def save_png(path_out: str, arr_rgb: np.ndarray):
    Image.fromarray(arr_rgb).save(path_out)

def save_gray_png(path_out: str, arr01: np.ndarray):
    img = (np.clip(arr01, 0, 1) * 255).astype(np.uint8)
    Image.fromarray(img).save(path_out)

def save_profile_plot(path_out: str, r_centers, p, q, alpha, rc, jsd, title=""):
    plt.figure(figsize=(7, 4))
    plt.plot(r_centers, p, label="Measured p(r)")
    plt.plot(r_centers, q, label=f"Ideal exp(-αr), α={alpha:.2f}")
    plt.xlabel("Normalized radius")
    plt.ylabel("Mass (normalized)")
    plt.title(f"{title} | RC={rc:.3f} (1-JSD), JSD={jsd:.3f}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path_out, dpi=150)
    plt.close()

def analyze_and_render_to_folder(img_rgb: np.ndarray,
                                 folder: str,
                                 summary: dict,
                                 mask_mode="subject",
                                 border_strength=2.2,
                                 nbins=60):
    """
    summary is the dict returned by analyze_path_quick() so visuals + numbers stay consistent.
    """
    ensure_dir(folder)

    # mask + mass map (use same compute_mask settings as batch)
    mask_bin, M = compute_mask(img_rgb, mode=mask_mode, border_strength=border_strength)
    mask_bin = np.asarray(mask_bin).astype(bool)
    M = np.asarray(M).astype(float)

    # Centers (IMPORTANT: use same "mass center" as batch cell = centroid_from_mask(mask_bin))
    fx, fy = frame_center(mask_bin.shape)
    sx, sy = centroid_from_mask(mask_bin)

    # --- visuals ---
    overlay = overlay_mask(img_rgb, mask_bin)
    overlay = draw_point(overlay, fx, fy, color=(0, 255, 255))  # frame center (cyan)
    overlay = draw_point(overlay, sx, sy, color=(255, 0, 0))    # mass center (red)

    rings_f = draw_rings(img_rgb, fx, fy, n=22, color=(220, 0, 0))
    rings_s = draw_rings(img_rgb, sx, sy, n=22, color=(220, 0, 0))

    # Save images
    save_png(os.path.join(folder, "01_original.png"), img_rgb)
    save_gray_png(os.path.join(folder, "02_mask.png"), mask_bin.astype(np.float32))
    save_png(os.path.join(folder, "03_overlay_centers.png"), overlay)
    save_png(os.path.join(folder, "04_rings_frame.png"), rings_f)
    save_png(os.path.join(folder, "05_rings_mass.png"), rings_s)

    # If mask was empty-ish, skip profile plots to avoid "blank chart" confusion
    mask_area = float(summary.get("mask_area", mask_bin.mean()))
    if mask_area < 0.001 or (summary.get("radial_label", "").startswith("MASK EMPTY")):
        return

    # --- profiles (for plots only) ---
    # Frame-centered
    pf, rav = radial_mass_profile(M, (fx, fy), nbins=nbins)
    pf = np.asarray(pf, dtype=float).ravel()
    rav = np.asarray(rav, dtype=float).ravel()

    af  = float(summary.get("alpha_f", np.nan))
    JSDf = float(summary.get("JSD_f", np.nan))
    RCf  = float(summary.get("RC_f", np.nan))
    qf = np.asarray(ideal_exp_profile(rav, af), dtype=float).ravel()

    # Mass-centered
    ps, _ = radial_mass_profile(M, (sx, sy), nbins=nbins)
    ps = np.asarray(ps, dtype=float).ravel()

    aS  = float(summary.get("alpha_s", np.nan))
    JSDs = float(summary.get("JSD_s", np.nan))
    RCs  = float(summary.get("RC_s", np.nan))
    qS = np.asarray(ideal_exp_profile(rav, aS), dtype=float).ravel()

    save_profile_plot(
        os.path.join(folder, "06_profile_frame.png"),
        rav, pf, qf, af, RCf, JSDf,
        title="Frame-centered radial compliance profile"
    )
    save_profile_plot(
        os.path.join(folder, "07_profile_mass.png"),
        rav, ps, qS, aS, RCs, JSDs,
        title="Mass-centered radial compliance profile"
    )

# ---- FIXED: Validate paths before running ----
if "paths" not in globals():
    raise RuntimeError("❌ 'paths' not defined. Run Cell 29 first.")

if not isinstance(paths, list):
    raise RuntimeError("❌ 'paths' is not a list. Run Cell 29 again.")

if len(paths) == 0:
    print("⚠️  WARNING: paths list is empty. No renders to generate.")
    print("   Re-run Cell 29 to upload images.")
    render_df = pd.DataFrame()
    display(render_df)

else:
    print(f"✓ Rendering outputs for {len(paths)} images...")

    render_rows = []
    for p in tqdm(paths):
        base = os.path.splitext(os.path.basename(p))[0]
        folder = os.path.join(OUT_DIR, base)

        try:
            # 1) compute metrics with canonical batch function
            s = analyze_path_quick(p, MASK_MODE, BORDER_STRENGTH, NBINS)

            # 2) render visuals using same image + same mask settings + same centers
            img = read_image_rgb(p)
            analyze_and_render_to_folder(
                img_rgb=img,
                folder=folder,
                summary=s,
                mask_mode=MASK_MODE,
                border_strength=BORDER_STRENGTH,
                nbins=NBINS
            )

            s["folder"] = folder
            render_rows.append(s)

        except Exception as e:
            render_rows.append({
                "filename": os.path.basename(p),
                "path": p,
                "mask_mode": MASK_MODE,
                "folder": folder,
                "error": f"{type(e).__name__}: {e}",
                "trace": traceback.format_exc(limit=8)
            })

    render_df = pd.DataFrame(render_rows)

    error_count = int(render_df.get("error", pd.Series()).notna().sum())
    print(f"\n✓ Render complete")
    print(f"  Total: {len(render_df)} images")
    print(f"  Errors: {error_count}")
    print(f"  Output directory: {OUT_DIR}")

    display(render_df.head(10))

20.

In [ ]:
ZIP_PATH = "/content/rca2_outputs.zip"

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, fns in os.walk(OUT_DIR):
        for fn in fns:
            full = os.path.join(root, fn)
            rel = os.path.relpath(full, OUT_DIR)
            zf.write(full, arcname=rel)

print("Zipped:", ZIP_PATH)
files.download(ZIP_PATH)

# Optional: also export a CSV of the render summaries
CSV_PATH = "/content/rca2_render_summaries.csv"
render_df.to_csv(CSV_PATH, index=False)
files.download(CSV_PATH)

21.

Radial collapse operates hierarchically.
Global field stabilization may dominate even when local subject forms remain radially coherent.
Subject radiality does not imply subject control.

This notebook is about a singular prompt → a singular default (hypothetically).
A priority behavior under minimal instruction.

Because once:
- you intentionally force asymmetry
- or introduce competing attractors
- or design an image to break the default

You are no longer measuring the default — you’re measuring recovery behavior.